# Project Assignment 2: Data Cleaning & Transformation

**Objective:**
In the previous assignment (Lab 5), you created a roadmap and identified the columns necessary for your research. In this assignment, you will execute that plan. You will perform the "dirty work" of data science: handling missing values, fixing data types, and transforming raw data into a format ready for analysis.

**Guidelines:**
* **Focus:** Only clean the data relevant to your research questions.
* **Justification:** You must explain *why* you chose a specific cleaning strategy (e.g., why did you drop a row vs. imputing the value?).
* **Reference:** Use the techniques covered in **Lecture 6** (Handling Missing Data, Transformations).

In [12]:
# --- Import Libraries ---
import pandas as pd
import numpy as np

# TODO: Load your dataset (use the filtered/subsetted data from Lab 5 if available)
df = pd.read_csv('../data/AB_NYC_2019.csv')
df.head()


,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
0,2539,Clean & quiet apt home by the park,2787,John,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,10/19/2018,0.21,6,365
1,2595,Skylit Midtown Castle,2845,Jennifer,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,5/21/2019,0.38,2,355
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Elisabeth,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,NaN,NaN,1,365
3,3831,Cozy Entire Floor of Brownstone,4869,LisaRoxanne,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,270,7/5/2019,4.64,1,194
4,5022,Entire Apt: Spacious Studio/Loft by central park,7192,Laura,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80,10,9,11/19/2018,0.10,1,0


## Part 1: Handling Missing Data (1.5 Points)

Your first task is to quantify the "missingness" in your dataset and apply a strategy to handle it. Refer to Lecture 6 for methods like `.isna()`, `.dropna()`, and `.fillna()`.

In [13]:
# TODO: df.isna().sum()

# Hint: df.isna().sum()

**Analysis of Missing Data:**
*Double click here to edit.*

1.  **Context & Bias:** Look at the specific rows with missing values. If you were to drop these rows, would you be systematically excluding a specific subgroup (e.g., a specific year, region, or demographic)? Explain how this might bias your analysis.

The reviews_per_month column contains missing values for listings with zero reviews. These missing values are not random; they occur specifically for properties that have never been reviewed. If these rows were dropped, it would systematically exclude new or inactive listings, potentially biasing the dataset toward more established and frequently reviewed properties.

2.  **Strategy Justification:** For the columns with missing data, explain *why* your chosen strategy (dropping vs. imputing) is the safest approach for your specific research question.
    * *Example:* "I am dropping 'Budget' because imputing an average would hide the distinction between indie and blockbuster films, which is central to my thesis."

Since listings with no reviews logically have zero reviews per month, I chose to impute missing values in the reviews_per_month column with 0. This preserves all listings in the dataset while maintaining accurate representation of review activity. Dropping these rows would unnecessarily reduce the dataset and distort the analysis.



In [14]:
# TODO: Execute your strategy.
df['reviews_per_month'] = df['reviews_per_month'].fillna(0)

# - Option A: Drop rows (df.dropna...)
# - Option B: Impute values (df.fillna...)

# Verify the cleanup:
df.isna().sum()


id                                    0
name                                 16
host_id                               0
host_name                            21
neighbourhood_group                   0
neighbourhood                         0
latitude                              0
longitude                             0
room_type                             0
price                                 0
minimum_nights                        0
number_of_reviews                     0
last_review                       10052
reviews_per_month                     0
calculated_host_listings_count        0
availability_365                      0
dtype: int64

## Part 2: Data Transformations (2.0 Points)

Raw data is rarely in the perfect format. In this section, you will convert data types, remove duplicates, and create new features (columns) that make analysis easier.

### 2.1 Fix Data Types
Ensure dates are actually timestamps and categories are recognized as such.

In [15]:
# TODO: Check current data types
df.dtypes 

df['last_review'] = pd.to_datetime(df['last_review'])


# TODO: Convert columns if necessary
# Example: df['date_col'] = pd.to_datetime(df['date_col'])
# Example: df['category_col'] = df['category_col'].astype('category')

### 2.2 Deduplication
Ensure you do not have duplicate records skewing your counts.

In [16]:
# TODO: Check for duplicates
df.duplicated().sum()

# TODO: Drop duplicates if they exist
# df = df.drop_duplicates()

np.int64(0)

### 2.3 Feature Engineering (Binning or Categories)
Create meaningful categories from continuous data (e.g., turning "Age" into "Age Group" or "Time" into "Rush Hour/Standard").

In [17]:

df['price_category'] = pd.cut(
    df['price'],
    bins=[0, 100, 200, 500, df['price'].max()],
    labels=['Low', 'Medium', 'High', 'Luxury']
)

df[['price', 'price_category']].head()


,price,price_category
0,149,Medium
1,225,High
2,150,Medium
3,89,Low
4,80,Low


## Part 3: Final Quality Check (1.5 Points)

Before you proceed to analysis in the next assignment, you must prove your dataset is clean.

In [18]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 48895 entries, 0 to 48894
Data columns (total 17 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   id                              48895 non-null  int64         
 1   name                            48879 non-null  str           
 2   host_id                         48895 non-null  int64         
 3   host_name                       48874 non-null  str           
 4   neighbourhood_group             48895 non-null  str           
 5   neighbourhood                   48895 non-null  str           
 6   latitude                        48895 non-null  float64       
 7   longitude                       48895 non-null  float64       
 8   room_type                       48895 non-null  str           
 9   price                           48895 non-null  int64         
 10  minimum_nights                  48895 non-null  int64         
 11  number_of_rev

**Final Reflection:**
*Double click here to edit.*

1.  **Trade-offs:** Compare your raw row count to your final row count. Explain why the benefits of having "clean" data outweigh the loss of sample size in your specific case.

Imputing reviews_per_month with 0 preserved all listings and avoided bias toward frequently reviewed properties. No rows were dropped, so the sample size remained intact.

2.  **Limitations:** Every data cleaning decision involves compromise. Describe one specific limitation your cleaning strategy has introduced. (e.g., "By removing outliers, I may be ignoring extreme but valid user behaviors.")

Converting price into price categories simplifies analysis but reduces detail, as continuous price variation is grouped into broader ranges.

In [19]:
# TODO: Save your clean dataset to a new CSV file
df.to_csv('../data/project_data_clean.csv', index=False)